# ODI to Databricks Migration: HR.TRG_DEP Load

**Conversion Timestamp:** 2024-07-30 12:00:00

This notebook loads data into the `HR.TRG_DEP` target table from `HR.DEPARTMENTS`.

In [ ]:
import datetime

dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type (F=Full, I=Incremental)")
dbutils.widgets.text("DATASOURCE_NUM_ID", "-1", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "ETL Process Widget ID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number")
dbutils.widgets.text("ETL_LAST_EXTRACT_TIME", datetime.datetime(1900, 1, 1).strftime('%Y-%m-%d %H:%M:%S'), "ETL Last Extract Time")
dbutils.widgets.text("ETL_CURRENT_EXTRACT_TIME", datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "ETL Current Extract Time")

# ETL Parameters

The following parameters control the execution of this notebook.

In [ ]:
display(spark.sql("""
  SELECT
    '${ETL_JOB_TYPE}' AS ETL_JOB_TYPE,
    '${DATASOURCE_NUM_ID}' AS DATASOURCE_NUM_ID,
    '${ETL_PROC_WID}' AS ETL_PROC_WID,
    '${ODI_SESS_NO}' AS ODI_SESS_NO,
    '${ETL_LAST_EXTRACT_TIME}' AS ETL_LAST_EXTRACT_TIME,
    '${ETL_CURRENT_EXTRACT_TIME}' AS ETL_CURRENT_EXTRACT_TIME
"""))

# Target Load - HR.TRG_DEP

In [ ]:
%sql
-- SCEN_TASK_NO in {10}
-- SCEN_TASK_NO in {20}
-- SCEN_TASK_NO in {30}
INSERT INTO workspace.hr.trg_dep
(
  DEPARTMENT_ID ,
  DEPARTMENT_NAME ,
  MANAGER_ID ,
  LOCATION_ID
)
SELECT
  DEPARTMENTS.DEPARTMENT_ID ,
  DEPARTMENTS.DEPARTMENT_NAME ,
  DEPARTMENTS.MANAGER_ID ,
  DEPARTMENTS.LOCATION_ID
FROM
  workspace.hr.departments AS DEPARTMENTS;

# Optimize Target

In [ ]:
%sql
-- Disable ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.hr.trg_dep ZORDER BY (DEPARTMENT_ID);

# Validation

In [ ]:
%sql
SELECT COUNT(*) AS total_rows FROM workspace.hr.trg_dep;

In [ ]:
%sql
SELECT
  DEPARTMENT_ID,
  DEPARTMENT_NAME,
  MANAGER_ID,
  LOCATION_ID
FROM
  workspace.hr.trg_dep
LIMIT 10;

# Conversion Notes

- The Oracle `/*+ APPEND PARALLEL */` hint has been removed as it is not applicable in Databricks Spark SQL for Delta tables.
- Schema and table names have been converted to lowercase and prefixed with `workspace.` (e.g., `HR.TRG_DEP` -> `workspace.hr.trg_dep`).
- `SCEN_TASK_NO` markers were retained as comments within the relevant SQL cell.
- An `OPTIMIZE ... ZORDER BY` statement has been added for the target table `workspace.hr.trg_dep` for performance, targeting `DEPARTMENT_ID` as a common join key.
- Standard Databricks widgets for ETL parameters have been included, although not directly used by this specific SQL statement.